# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata # This is an object, not a dictionary

print(f"{metadata.name}: {metadata.description}")
print(f"Dataset ID: {metadata.identifier}")
print(f"Authors: {', '.join([str(au['@id']) for au in getattr(metadata, 'author', [])])}")
print(f"Date published: {getattr(metadata, 'datePublished', 'Unknown')}")
print(f"Keywords: {', '.join(getattr(metadata, 'keywords', []))}")
print(f"Croissant schema version: {getattr(metadata, 'conformsTo', 'Unknown')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets with their @id and fields
record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets declared in the Croissant metadata (top-level 'recordSet' is empty).")
    print("Let's try to infer available record sets from distributions.")
    # Try to access any available record set from data and schema (more advanced usage)
    distribution_ids = [d['@id'] for d in getattr(metadata, 'distribution', [])]
    print(f"Distribution @ids: {distribution_ids}")
    # We'll just try to use the default record set name 'MainTable' (most frequent convention)
    inferred_record_sets = []
    for record_set in dataset.dataset['@graph'] if hasattr(dataset, 'dataset') and dataset.dataset else []:
        if record_set.get('@type') == 'RecordSet':
            rs_id = record_set.get('@id')
            inferred_record_sets.append(rs_id)
            print(f"Found RecordSet: {rs_id}")
            fields = record_set.get('field') or []
            if isinstance(fields, dict):
                fields = [fields]
            print("  Fields:")
            for field in fields:
                if isinstance(field, dict):
                    print(f"    {field.get('@id')} ({field.get('name', 'Unknown name')})")
                else:
                    print(f"    {field}")
    if not inferred_record_sets:
        # Some croissant datasets use a single record set accessible via the dataset
        print("No explicit record sets — trying to load records with record_set=None (default)...")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        if 'field' in rs:
            fields = rs['field']
            if not isinstance(fields, list):
                fields = [fields]
            print("  Fields (by @id):")
            for field in fields:
                if isinstance(field, dict):
                    print(f"    {field.get('@id')}")
                else:
                    print(f"    {field}")
        else:
            print("  No fields declared.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Since recordSet might be empty in top-level metadata, attempt to infer the correct @id
from itertools import islice

# Attempt to discover a valid record set @id for this dataset
record_set_ids = []
if hasattr(dataset, 'record_sets'):
    for rs in dataset.record_sets:
        if '@id' in rs:
            record_set_ids.append(rs['@id'])

if not record_set_ids:
    # Try to infer from the schema graph
    # This will only work if dataset.dataset is available and in JSON-LD @graph form
    if hasattr(dataset, 'dataset') and dataset.dataset and '@graph' in dataset.dataset:
        for node in dataset.dataset['@graph']:
            if node.get('@type') == 'RecordSet':
                record_set_ids.append(node['@id'])

if not record_set_ids:
    print("Could not determine any RecordSet @id; attempting to load records with record_set=None.")
    record_set_ids = [None]

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(islice(dataset.records(record_set=record_set_id), 1000))  # sample up to 1000 rows
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set {record_set_id}: shape {df.shape}")
        print("Columns:", df.columns.tolist())
        display(df.head())
    except Exception as e:
        print(f"Could not load data for record set {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select one record set for analysis
selected_record_set_id = record_set_ids[0]
df = dataframes[selected_record_set_id]

# Try to automatically detect a numeric field
numeric_field = None
for col in df.columns:
    # Try to detect a numeric type by attempting conversion
    try:
        sample = pd.to_numeric(df[col], errors='coerce')
        if sample.notnull().sum() > 0:
            numeric_field = col
            break
    except Exception:
        continue

if numeric_field is None:
    print("No numeric field detected for EDA!")
else:
    print(f"Selected numeric field: {numeric_field}")
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].quantile(0.9)  # use 90th percentile as a threshold example
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (top 10%):")
    display(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to automatically detect a string/categorical column for grouping
    group_field = None
    for col in df.columns:
        if col != numeric_field and df[col].dtype == object:
            n_unique = df[col].nunique()
            if 1 < n_unique < max(20, len(df) // 10):
                group_field = col
                break

    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field} (top 5 groups):")
        display(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(10,6))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, we loaded the FAIR^2 dataset: **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** using the Croissant schema and the `mlcroissant` library.
* We explored metadata, inspected available record sets and fields by their `@id`, loaded the main record set to pandas DataFrames for processing, and performed basic exploratory data analysis and visualization.
* The specific numeric and categorical fields used in EDA were automatically detected. For detailed studies or domain-specific analysis (e.g., protein coverage, abundance), refer to the [dataset schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and map `@id` fields accordingly.
* This notebook provides a reusable workflow for other Croissant datasets: remember to validate field data types and check schema documentation for domain-specific analyses.